# Dynamic Pricing Env with Inventory Constraints

> Static dynamic pricing environment with inventory constraints. Through which the decision affect all folowing periods.

In [ ]:
#| default_exp envs.pricing.dynamic_inventory

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from abc import ABC, abstractmethod
from typing import Union, Tuple, Literal

from ddopai.utils import Parameter, MDPInfo
from ddopai.dataloaders.base import BaseDataLoader
from ddopai.loss_functions import pinball_loss, quantile_loss
from ddopai.envs.pricing.base import BasePricingEnv

import gymnasium as gym

import numpy as np
import time

In [ ]:
# | export
class DynamicPricingInvEnv(BasePricingEnv):
    """
    Class implementing the dynamic pricing and learning problem with inventory constraints, working for the single- and multi-item case.
    If alpha and beta are scalars and they are multiple SKUs, then the same parameters are used for all SKUs.
    If alpha and beta are arrays, then they should have the same length as the number of SKUs.
    Num_SKUs can be set as parameter or inferrred from the DataLoader.
    
    """
    def __init__(self,
        alpha: Union[np.ndarray, Parameter, int, float] = 1.0, # market size per SKUs
        beta: Union[np.ndarray, Parameter, int, float] = 0.5, # price elasticity per SKUs
        p_bound_low: Union[np.ndarray, Parameter, int, float] = 0.0, # lower price bound per SKUs
        p_bound_high: Union[np.ndarray, Parameter, int, float] = 1.0, # upper price bound per SKUs
        dataloader: BaseDataLoader = None, # dataloader TODO: replace with pricing dataloader
        num_SKUs: Union[np.ndarray, Parameter, int, float] = None, # number of SKUs
        gamma: float = 1, # discount factor
        
        inv: Union[np.ndarray, Parameter, int, float] = 100, # inventory per SKUs
        nb_features: int = 1, # number of features
        covariance: Union[np.ndarray, Parameter, int, float] = 1, # standard deviation of the features
        noise_std: Union[np.ndarray, Parameter, int, float] = 1, # standard deviation of the noise
        function_form: Union[np.ndarray, Parameter, str] = "linear", # functional form of the demand function
        
        horizon_train: int | str = "use_all_data", # if "use_all_data" then horizon is inferred from the DataLoader
        postprocessors: list[object] | None = None, # default is empty list 
        mode: str = "train", 
        return_truncation: str = False # TODO:Why is this a string?
        ) -> None:

        self.print=False
        
        num_SKUs = dataloader.num_units if num_SKUs is None else num_SKUs
        
        if not isinstance(num_SKUs, int):
            raise ValueError("num_SKUs should be an integer.")
        if not alpha.shape==beta.shape:
            raise ValueError("alpha and beta should have the same shape.")
        self.set_param("num_SKUs", num_SKUs, new=True)
        
        self.set_param("alpha", alpha, shape=alpha.shape, new=True)
        self.set_param("beta", beta, shape=beta.shape, new=True)
        self.set_param("p_bound_low", p_bound_low, shape=(num_SKUs,), new=True)
        self.set_param("p_bound_high", p_bound_high, shape=(num_SKUs,), new=True)
        
        self.set_param("nb_features", nb_features, new=True)
        if isinstance(covariance, np.ndarray):
            self.set_param("covariance", covariance, shape=covariance.shape, new=True)
        else:
            self.set_param("covariance", covariance,  new=True)
        if isinstance(noise_std, np.ndarray):
            self.set_param("noise_std", noise_std, shape=noise_std.shape, new=True)
        else:
            self.set_param("noise_std", noise_std, new=True)
        self.set_param("function_form", function_form, new=True)
        
        self.set_param("inv", inv[0], inv[0].shape, new=True)
        relative_inv = inv[0].copy()
        relative_inv[-1]= 1.0# np.float64(1.0)
        self.set_param("relative_inv", relative_inv, relative_inv.shape, new=True)
        self.set_param("inv_per_episode", inv, inv.shape, new=True)
        self.set_param("horizon_train", horizon_train, new=True)
        
        self.set_param("info_history", {}, new=True)
        X_shape = dataloader.X_shape
        X_shape = list(X_shape)
        X_shape[-1] += 1
        X_shape = tuple(X_shape)
        self.set_observation_space(X_shape)
        self.set_action_space(dataloader.Y_shape, low = self.p_bound_low, high = self.p_bound_high)
        
        
        mdp_info = MDPInfo(self.observation_space, self.action_space, gamma=gamma, horizon=horizon_train)
        
        super().__init__(mdp_info=mdp_info,
                         postprocessors=postprocessors,
                         mode=mode, return_truncation=return_truncation,
                         dataloader=dataloader,
                         horizon_train=horizon_train)
        
    def step_(self,
              action: np.ndarray # prices)
                ) -> Tuple[np.ndarray, float, bool, bool, dict]:
        """
        Step function implementing the dynamic pricing and learning problem. Note that the dataloader will return an observation and a demand function.
        """
        if action.ndim == 2 and action.shape[0] == 1:
            action = np.squeeze(action, axis=0)
            
        
        
        terminated = False
        observation, reward_functions = self.get_observation() 

        demand_per_SKU, demand_per_SKU_noise_free = [], []
        for idx, (reward_function, x, a) in enumerate(zip(reward_functions, observation, action)):
            x = x[1:] 
            demand, demand_noise_free = reward_function(x, a)
            if np.divide(demand, self.inv[idx]) >= self.relative_inv[idx]:
                demand = self.relative_inv[idx]*self.inv[idx]
                
                if np.divide(demand_noise_free, self.inv[idx]) >= self.relative_inv[idx]:
                    demand_noise_free = self.inv[idx]*self.relative_inv[idx]
                self.relative_inv[idx] = 0
            else:
                self.relative_inv[idx] -= demand/self.inv[idx]
            demand_per_SKU.append(demand)
            demand_per_SKU_noise_free.append(demand_noise_free)
            
        demand_per_SKU = np.array(demand_per_SKU)
        demand_per_SKU_noise_free = np.array(demand_per_SKU_noise_free)
        reward_per_SKU = demand_per_SKU * action
        reward_per_SKU_noise_free = demand_per_SKU_noise_free * action
        reward = np.sum(reward_per_SKU)
        
        if np.all(self.relative_inv == 0):
            terminated = True
        info = dict(
            inv=self.inv.copy()*self.relative_inv.copy(),
            demand=demand_per_SKU.copy(),
            demand_per_SKU_noise_free=demand_per_SKU_noise_free.copy(),
            action=action.copy(),
            reward_per_SKU=reward_per_SKU.copy(),
            reward_per_SKU_noise_free=reward_per_SKU_noise_free.copy()
        )
        
        self.info_history[len(self.info_history)] = info
        truncated = self.set_index()
        
        if truncated:

            if self.mode == "test" or self.mode == "val":
                observation= None
            else:
                observation, _ = self.get_observation()

            return observation, reward, terminated, truncated, info
        
        else:

            # TODO: check if this is correct since we are not interested in the next period 
            if self.print:
                print("next_period:", self.index+1)
                print("next observation:", observation)
                time.sleep(3)

            return observation, reward, terminated, truncated, info
    def new_episode(self, epoch) -> None:
        """
        Function to reset the environment.
        """
        if epoch > 0:
            feature_index = epoch % self.covariance.shape[0]
            X = np.random.multivariate_normal(np.ones(self.nb_features[0]-1), self.covariance[feature_index]*np.eye(self.nb_features[0]-1), size=self.horizon_train+1)
            X = np.hstack((np.ones((self.horizon_train+1, 1)), X))
            X = X.reshape(-1, 1, self.nb_features[0])
            
            epsilon = np.random.normal(0, self.noise_std[feature_index], size=self.horizon_train+1)
            
            parameter_index = epoch % self.alpha.shape[0]
            function_form_index = epoch % self.function_form.shape[0]
            inv_index = epoch % self.inv_per_episode.shape[0]
            self.inv = self.inv_per_episode[inv_index]
            self.relative_inv = self.inv_per_episode[inv_index].copy()
            self.relative_inv[-1] = 1.0
            self.dataloader.update_parameters(X=X, epsilon=epsilon, alpha=self.alpha[parameter_index], beta=self.beta[parameter_index], function_form=self.function_form[function_form_index])
            self.info_history = {}
            self.reset()
        return None
    
    def get_observation(self):
        """
        Function to get the observation from the dataloader.
        """
        observation, reward_functions = self.dataloader[self.index]
        observation = np.concatenate((self.relative_inv, observation), axis=1)
        
        return observation, reward_functions
    
    
    def get_info_history(self) -> dict:
        """
        Function to return the history of the environment.
        """
        return self.info_history